# 03 · EDA por dimensión y ronda

**Objetivo:** resumir cada variable sin perder interpretación ni mezclar rondas. Las gráficas futuras conservarán paleta BrBG y etiquetado de categorías. Este notebook contiene un primer EDA reproducible, no la versión exhaustiva CP2.


In [ ]:
from tesis_enasem.config import RAW_DIR, ROUND_TO_YEAR
from tesis_enasem.io import load_stata
from tesis_enasem.panel import make_person_id, add_year, filter_population
from tesis_enasem.eda import summarize_by_wave, availability_by_wave
from tesis_enasem.variables import GRUPOS
from tesis_enasem.missing import numeric_enasem
import matplotlib.pyplot as plt
import pandas as pd

raw=add_year(load_stata(RAW_DIR/'simpleMHAS.dta',preserve_missing=True),ROUND_TO_YEAR)
df=filter_population(raw, min_age=60, interview_types=(1,2))


## Filtro editable de análisis exploratorio
Cambiar variables de control y ejecutar la siguiente celda para generar resultados de la subpoblación seleccionada. Los conteos son **observaciones persona-ronda**, no individuos distintos a lo largo del seguimiento.


In [ ]:
SEXO = None       # None: todos, 1: hombres, 2: mujeres
URBANO = None     # None: todos, 0: localidades <100 mil, 1: 100 mil+
ANOS = None       # None: todas las olas, o [2012,2015,2018,2021]

sub=df.copy()
if SEXO is not None: sub=sub[numeric_enasem(sub['sexo']).eq(SEXO)]
if URBANO is not None: sub=sub[numeric_enasem(sub['urbano']).eq(URBANO)]
if ANOS is not None: sub=sub[sub['ano'].isin(ANOS)]
print(f'{len(sub):,} persona-rondas, {len(sub.groupby(["ano"]))} olas')


## Gráfico inicial por variable y año
En casos categóricos usar barras; para `imp_neto` y `bmi_imp` comparar distribuciones con histogramas/densidades en el CP2 ampliado.


In [ ]:
VARIABLE='n_enf'  # escoger cualquier numérica de GRUPOS
tab=summarize_by_wave(sub, VARIABLE)
display(tab)
fig,ax=plt.subplots(figsize=(8,4))
colors=plt.cm.BrBG([.08,.25,.4,.6,.75,.92])
ax.bar(tab['ano'].astype(str),tab['mean'],color=colors[:len(tab)])
ax.set(xlabel='Año nominal de ronda', ylabel=f'Media de {VARIABLE}',title=f'{VARIABLE} · medias por año')
ax.grid(axis='y',alpha=.25)
plt.show()


## Inventario de todas las variables seleccionadas


In [ ]:
variables=[v for group in GRUPOS.values() for v in group if v in sub.columns]
display(availability_by_wave(sub,variables).head(40))


**No confundir disponibilidad desigual con tendencia clínica/económica.** Antes de interpretación final, revisar distribuciones y ponderación por ronda.
